# Day 1 — Track Selection, Architecture Doc & Backend Kick-off

---

Welcome to the final section. **Class time is only 75 min/day for 2 days** — real capstone work happens **on your own time over ~1–2 weeks**.

Today: pick a track, write the architecture doc, and get a bare backend deployed. Nothing AI-related yet. This is the plumbing.


## Part 1 — Pick your track (mandatory, commit today)

Pick ONE. Switching mid-way is expensive.

**Track A — AI SaaS Product**
Consumer- or team-facing web product. Would fit in a Product Hunt launch.
Good if you want a **portfolio piece non-technical people understand**.

**Track B — Enterprise Knowledge AI**
Internal chatbot over a private corpus. Recruiter-friendly for enterprise roles.
Good if you already have a **domain you know well** (legal / healthcare / finance / education).

**Track C — AI Automation Platform**
Multi-agent business-workflow automation. Highest-wow demo when executed well.
Good if you loved **Section 7 (Agents)** and want to go deeper.

**Signals to pick:**
- Which section did you enjoy most? (S6 → B, S7 → C, S9 → A)
- Do you have a real use case in your life?
- What roles are you interviewing for?


## Part 2 — Scope with one paragraph

Fill the blanks. Post it in your capstone repo README today.

> *My project is a **[track]** that helps **[user]** do **[task]** by using **[main AI feature]**. Users interact via **[interface]**. Success looks like **[measurable outcome]**.*

Example (Track B):

> *My project is an Enterprise Knowledge AI that helps HR staff answer employee questions about benefits by using RAG over our 200-page handbook and past HR tickets. Users interact via a Slack bot. Success = ≥80% questions answered without a human handoff.*


## Part 3 — Architecture doc (`docs/ARCHITECTURE.md`)

9 sections — use `ARCHITECTURE_TEMPLATE.md` in this folder:

1. **Problem** (2 sentences)
2. **Users** (who, how many)
3. **User flow** (3-5 steps)
4. **Architecture diagram**
5. **Stack** — libraries / services with reason
6. **Data** — sources + ingestion cadence
7. **Cost model** — cost per active user
8. **Risks & unknowns**
9. **Milestones** — day plan


### Architecture diagram — the shape per track

**Track A — AI SaaS**
```
[Web app] → [Gateway] → [FastAPI] → [Postgres]
                          │
                          ├─► [LLM API]
                          ├─► [Chroma / pgvector]
                          └─► [Stripe / auth]
```

**Track B — Enterprise Knowledge AI**
```
[Confluence / SharePoint / DB] ──ingestion──► [Chroma]
                                                  │
[Slack / web UI] → [FastAPI] → [RAG pipeline] ────┤
                                                  └─► [LLM API]
```

**Track C — AI Automation**
```
[Trigger: email / webhook / cron] → [FastAPI]
                                       │
                                       ▼
                        [LangGraph agent workflow]
                             │  ├─► search
                             │  ├─► extract
                             │  ├─► decide
                             │  └─► act (email, DB, Slack)
                             ▼
                       [HITL approval queue]
```

Draw yours today (Excalidraw / draw.io / whiteboard photo). Commit as `docs/architecture.png`.


### Cost model

Use `main.py` in this folder. Pick real numbers:
- Expected DAU (start small — 10 or 100)
- Queries per user per day
- Avg tokens per query
- Your LLM price / 1M tokens

Compute daily + monthly. **Write it in the doc.** If it's >$100/mo for demo, shrink the model or add caching now.


### Milestones — the fresher plan (adapt as needed)

Class is 2 days, but real work is spread across ~1-2 weeks. Suggested checkpoints:

- **Week 1**: Backend + auth + `/healthz` + CI green + first Render deploy (Day 1 tasks)
- **Week 1 mid**: RAG pipeline works locally + on deployment (Day 2 morning tasks)
- **Week 2 start**: Agent workflow works end-to-end (Day 2 tasks)
- **Week 2 mid**: Full observability + cost cap + load-test (Day 2 tasks)
- **Week 2 end**: Demo video + cost analysis + portfolio page (Day 2 final)

Create GitHub Issues for each. Check off as you go.


## Part 4 — Non-negotiable rules for the capstone

- **Commit every day.** No 4-day `main` silences.
- **Deploy something early.** Hello-World on day 1 beats monolith on day 6.
- **Real data over synthetic.** 5 real PDFs > 100 fake ones for a demo.
- **One demo user story.** Nail one, don't demo 6.
- **Stuck for >2 hours? Ask.** Don't spin.


## Part 5 — Backend + auth kick-off (start today)

The minimum backend for capstone success:

- `POST /auth/register` — create user (hashed password)
- `POST /auth/login` — return JWT
- `GET /me` — current user (JWT protected)
- `GET /healthz` — liveness check
- `GET /` — friendly root (JSON is fine)

**Nothing AI yet.** Get the plumbing to a green CI + deployed URL first.


### Directory layout — the shape that scales

```
capstone/
├── app/
│   ├── main.py            # FastAPI app + route mounts
│   ├── config.py          # env-var loading (pydantic-settings)
│   ├── db.py              # SQLAlchemy setup
│   ├── models.py          # User, Session, UsageRecord
│   ├── auth.py            # JWT + password hashing (copy Section 2)
│   ├── routes/
│   │   ├── auth.py
│   │   ├── health.py
│   │   └── ...            # add RAG next class, agent next class
│   └── services/
├── tests/
├── Dockerfile             # from Section 9 Day 1
├── requirements.txt
└── .github/workflows/ci.yml   # from Section 9 Day 2
```

Split by responsibility. 30 min today, saves days later.


### Database — two-file setup

Local: **SQLite** (zero-config). Production (Render): **Postgres** (managed cheap tier).

```python
# app/db.py
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base
import os

DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///./local.db")
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False} if DATABASE_URL.startswith("sqlite") else {},
)
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()
```

Same code works for both — just change `DATABASE_URL`.


### Auth — copy Section 2 Day 9 verbatim

- `hash_password(plain)` → bcrypt
- `verify_password(plain, hashed)`
- `create_access_token(sub)`
- `get_current_user(token = Depends(oauth2))`

**Don't reinvent.** See `main.py` in this folder for a self-contained starter.


### The one integration test you must have today

```python
# tests/test_smoke.py
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

def test_healthz():
    r = client.get("/healthz")
    assert r.status_code == 200

def test_auth_flow():
    client.post("/auth/register", json={"email": "a@a.com", "password": "pw12345678"})
    tok = client.post("/auth/login", json={"email": "a@a.com", "password": "pw12345678"}).json()["token"]
    r = client.get("/me", headers={"Authorization": f"Bearer {tok}"})
    assert r.status_code == 200
```

**This test green in CI is your Day-1 bar.** Proves the loop works before AI complexity lands.


### Deploy TODAY

Push to `main`. Deploy to Render (Section 9 Day 3 walkthrough).

**Live URL with `/docs` reachable before you sleep tonight.** Every subsequent day is faster.


## Day 1 deliverables (before you sleep)

- [ ] Track chosen + committed to a fresh GitHub repo README
- [ ] One-paragraph project pitch in README
- [ ] `docs/ARCHITECTURE.md` — all 9 sections filled
- [ ] `docs/architecture.png`
- [ ] GitHub Issues for the milestone plan
- [ ] Auth routes + `/healthz` + `/me` working locally
- [ ] `test_healthz` + `test_auth_flow` passing
- [ ] `Dockerfile` + `.github/workflows/ci.yml` — green
- [ ] Deployed to Render, public `/docs` reachable


## Recap

- Pick a track. Commit. Write the pitch.
- Fill out `ARCHITECTURE.md`. Model the cost upfront.
- **Deploy the skeleton today** — plumbing before AI.
- Two working tests in CI = your gate to Day 2.
- **Next class:** RAG → Agent → deploy → observe → demo.
